# Module 6: Medi-Cal Primary-Language Data Cleaning

**Student:** Jonathan Guo  
**Goal:** Prepare DHCS Medi-Cal certified-eligibles data for analysis and a future dashboard comparing counties and primary-language groups.  
**Business question:** Which California counties and primary-language groups have the largest numbers of Medi-Cal certified eligibles and may warrant closer review for language-access planning?

The rules were written first in `plan.md`. The notebook calls the same reusable functions as `clean_data.py`, so the script and notebook cannot silently implement different cleaning logic.

## Decisions before cleaning

- No rows will be dropped. Unusual records are retained and flagged.
- Disclosure-suppressed counts remain missing; they are not zero and are not imputed.
- `Unknown` and `Missing/Unknown` remain separate source categories.
- Statewide rows are retained but kept out of county sums to prevent double counting.
- Invalid-county records are retained but flagged because their county cannot be inferred safely.
- Preliminary (`P`) and final (`F`) records remain distinguishable.

In [7]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

from clean_data import (
    DEFAULT_INPUT, build_decision_log, build_missingness_table,
    build_record_checks, clean_data, load_source, profile_source,
    run_pipeline, validate
)

pd.set_option('display.max_colwidth', 80)
raw = load_source(DEFAULT_INPUT)
source_profile = profile_source(raw, DEFAULT_INPUT)
cleaned, validation = run_pipeline(DEFAULT_INPUT)

Rows: 251,535 -> 251,535
Missing total_eligibles: 94,321 source nulls -> 94,324 cleaned missing values
All 9 required checks passed.
cleaned_full: output/by-primary-language-certified-eligibles-cleaned.csv
sample: cleaned_data_sample.csv
source_profile: evidence/source_profile.json
validation: evidence/validation_summary.json
missingness: evidence/missingness_before_after.csv
decisions: evidence/cleaning_decision_log.csv
record_checks: evidence/record_checks.csv


## 1. Inspect the source before transformations

One source row represents a unique month × county/geography × primary-language observation. The proposed key is checked separately from exact duplicates.

In [8]:
print('Source shape:', raw.shape)
print('Month range:', source_profile['month_min'], 'to', source_profile['month_max'])
print('Exact duplicates:', source_profile['exact_duplicate_rows'])
print('Repeated-key excess rows:', source_profile['duplicate_key_rows_excess'])
display(raw.head())
display(pd.DataFrame({'dtype_before': raw.dtypes.astype(str), 'missing_before': raw.isna().sum()}))

Source shape: (251535, 9)
Month range: 2010-01 to 2026-08
Exact duplicates: 0
Repeated-key excess rows: 0


,Month of Eligibility,County,Language,Total Eligibles,Annotation Code,Annotation Description,Status,Date of Data,Difference
0,2026-08,Alameda,ASL,132,<NA>,<NA>,P,11AUG2026,<NA>
1,2026-08,Alameda,Arabic,"2,724",<NA>,<NA>,P,11AUG2026,<NA>
2,2026-08,Alameda,Armenian,35,<NA>,<NA>,P,11AUG2026,<NA>
3,2026-08,Alameda,Cambodian,715,<NA>,<NA>,P,11AUG2026,<NA>
4,2026-08,Alameda,Cantonese,"23,765",<NA>,<NA>,P,11AUG2026,<NA>


,dtype_before,missing_before
Month of Eligibility,string,0
County,string,0
Language,string,0
Total Eligibles,string,94321
Annotation Code,string,157211
Annotation Description,string,157011
Status,string,0
Date of Data,string,138139
Difference,string,251535


## 2. Review rules and evidence of each change

In [9]:
display(build_decision_log())
display(build_missingness_table(raw, cleaned))

,rule,reason,check
0,Preserve all rows,Unusual does not mean erroneous.,Row count before equals after.
1,Parse month as YYYY-MM,"Month is a time field, not free text.",Zero failed meaningful conversions.
2,Trim outer text whitespace,Remove formatting artifacts only.,"2,573 language values change; categories are reviewed."
3,Keep Unknown separate from Missing/Unknown,No data dictionary evidence supports merging them.,Both categories remain present.
4,Convert totals to nullable integers,Counts require numeric operations while allowing suppression.,No meaningful conversion failures; all values are whole and nonnegative.
5,Do not impute suppressed totals,"Suppression is unknown, not zero.",Missing totals align exactly with suppression flags.
6,Convert '.' date marker to missing,The marker is not a calendar date.,Zero failures among meaningful date strings.
7,Retain and flag invalid county,A county cannot be inferred safely.,Four records retained with has_valid_county=False.
8,Retain and flag statewide rows,They are useful totals but would double count county aggregation.,200 records retained with is_statewide=True.
9,Preserve F/P status and add label,Preliminary and final releases must be distinguishable.,All statuses map to Final or Preliminary.


,source_column,clean_column,missing_before,missing_after,explanation
0,Month of Eligibility,month_of_eligibility,0,0,No missingness change.
1,County,county,0,0,No missingness change.
2,Language,language,0,0,No missingness change.
3,Total Eligibles,total_eligibles,94321,94324,Three whitespace-only complementary-suppression cells become missing.
4,Annotation Code,annotation_code,157211,157211,No missingness change.
5,Annotation Description,annotation_description,157011,157011,No missingness change.
6,Status,status,0,0,No missingness change.
7,Date of Data,date_of_data,138139,141875,Publisher '.' markers are standardized to missing.
8,Difference,<dropped>,251535,<NA>,Dropped because every source value is missing; no rows dropped.


In [10]:
print('Language values changed only by outer-whitespace trim:', validation['language_values_changed_by_trim'])
print('County values changed by trim:', validation['county_values_changed_by_trim'])
print('Language categories after cleaning:', validation['clean_language_categories'])
print('Unknown categories remain distinct:')
display(cleaned.loc[cleaned['language'].isin(['Unknown', 'Missing/Unknown']), 'language'].value_counts())

Language values changed only by outer-whitespace trim: 2573
County values changed by trim: 0
Language categories after cleaning: 34
Unknown categories remain distinct:


language
Missing/Unknown    11572
Unknown             2573
Name: count, dtype: Int64

## 3. Validate numeric values, dates, distributions, and suspicious records

Because suppressed cells are unknown rather than zero, any sum below is a **lower bound** over reported cells. The record count must accompany the sum.

In [11]:
metrics = [
    'month_parse_failures', 'total_parse_failures_meaningful',
    'date_parse_failures_meaningful', 'negative_total_rows',
    'numeric_record_count', 'numeric_min', 'numeric_max',
    'numeric_sum_all_rows_lower_bound'
]
display(pd.Series({name: validation[name] for name in metrics}, name='value').to_frame())
display(pd.Series(validation['numeric_quantiles'], name='total_eligibles').to_frame())

,value
month_parse_failures,0
total_parse_failures_meaningful,0
date_parse_failures_meaningful,0
negative_total_rows,0
numeric_record_count,157211
numeric_min,11
numeric_max,16051565
numeric_sum_all_rows_lower_bound,4914166969


,total_eligibles
0.0,11.0
0.25,36.0
0.5,144.0
0.75,1433.0
0.95,62060.5
0.99,323132.3
1.0,16051565.0


In [12]:
print('Suppressed rows:', validation['suppressed_rows'])
print('Statewide rows:', validation['statewide_rows'])
print('Invalid-county rows:', validation['invalid_county_rows'])
print('Preliminary rows:', validation['preliminary_rows'])
display(cleaned.nlargest(10, 'total_eligibles')[['month_of_eligibility', 'county', 'language', 'total_eligibles', 'status', 'is_statewide']])
display(cleaned.loc[~cleaned['has_valid_county']])

Suppressed rows: 94324
Statewide rows: 200
Invalid-county rows: 4
Preliminary rows: 16526


,month_of_eligibility,county,language,total_eligibles,status,is_statewide
53498,2023-05-01,STATEWIDE,STATEWIDE,16051565,F,True
54840,2023-04-01,STATEWIDE,STATEWIDE,15981071,F,True
56178,2023-03-01,STATEWIDE,STATEWIDE,15897515,F,True
57520,2023-02-01,STATEWIDE,STATEWIDE,15798053,F,True
58860,2023-01-01,STATEWIDE,STATEWIDE,15717809,F,True
52156,2023-06-01,STATEWIDE,STATEWIDE,15672367,F,True
49479,2023-08-01,STATEWIDE,STATEWIDE,15638672,F,True
50819,2023-07-01,STATEWIDE,STATEWIDE,15612841,F,True
60191,2022-12-01,STATEWIDE,STATEWIDE,15586919,F,True
61525,2022-11-01,STATEWIDE,STATEWIDE,15474582,F,True


,source_row_number,month_of_eligibility,county,language,total_eligibles,annotation_code,annotation_description,status,status_label,date_of_data,is_suppressed,is_statewide,has_valid_county,is_preliminary
15421,15423,2025-09-01,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,P,Preliminary,2026-08-11,True,False,False,True
223773,223775,2011-12-01,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,F,Final,NaT,True,False,False,False
223774,223776,2011-12-01,Invalid County Code,Spanish,<NA>,1,Cell suppressed for small numbers,F,Final,NaT,True,False,False,False
237843,237845,2010-12-01,Invalid County Code,English,<NA>,1,Cell suppressed for small numbers,F,Final,NaT,True,False,False,False


The largest observations are statewide totals, so their size is expected and is not corrected. They are flagged to prevent accidental addition to county-level rows. Invalid county codes are retained because no trustworthy replacement county is available.

## 4. Five predeclared record checks

Each expectation comes from `plan.md` and is stated before the actual cleaned value is evaluated.

In [13]:
record_checks = build_record_checks(raw, cleaned)
display(record_checks)
assert record_checks['matches_expected'].all()

,check,source_row_number,original_count,original_count_is_missing,original_count_repr,original_county,original_language,original_annotation_code,expected_before_checking,actual_count,actual_count_is_missing,actual_county,actual_language,actual_is_suppressed,actual_has_valid_county,matches_expected
0,regular numeric record,2,132,False,'132',Alameda,ASL,<NA>,Count converts to the same integer; row remains unsuppressed.,132,False,Alameda,ASL,False,True,True
1,small-cell suppression,10,<NA>,True,<NA>,Alameda,Hebrew,1,Count remains missing and is_suppressed is True.,<NA>,True,Alameda,Hebrew,True,True,True
2,complementary suppression,34,,False,' ',Alpine,English,2,Whitespace/blank count becomes missing and is_suppressed is True.,<NA>,True,Alpine,English,True,True,True
3,legacy language whitespace,19296,"2,255",False,"'2,255'",Alameda,Unknown,<NA>,"Language becomes Unknown, remains distinct from Missing/Unknown.",2255,False,Alameda,Unknown,False,True,True
4,invalid county code,15423,<NA>,True,<NA>,Invalid County Code,English,1,Row is retained and has_valid_county is False.,<NA>,True,Invalid County Code,English,True,False,True


## 5. Reconciliation and reproducibility

In [14]:
display(pd.Series(validation['required_checks'], name='passed').to_frame())
print(f"Rows reconciled: {validation['rows_before']:,} -> {validation['rows_after']:,}; difference {validation['row_difference']}.")
print(f"Raw and cleaned reported-cell sums: {validation['raw_numeric_sum']:,} -> {validation['numeric_sum_all_rows_lower_bound']:,}.")
print(f"Latest final month: {validation['latest_final_month']}; valid county numeric records: {validation['latest_final_valid_county_numeric_records']:,}.")
print('Submission sample rows:', len(pd.read_csv('cleaned_data_sample.csv')))

,passed
row count preserved,True
month conversions succeeded,True
meaningful numeric conversions succeeded,True
meaningful date conversions succeeded,True
expected key remains unique,True
status values are recognized,True
counts are nonnegative,True
missing counts match suppression,True
numeric sum unchanged,True


Rows reconciled: 251,535 -> 251,535; difference 0.
Raw and cleaned reported-cell sums: 4,914,166,969 -> 4,914,166,969.
Latest final month: 2025-08; valid county numeric records: 870.
Submission sample rows: 50
